# step back prompt
以小台指期一小時為單位  
out_kine_SBP_30min 是30分線 
針對k線走勢評估接下來的漲跌

In [ ]:
# %pip install mplfinance
# %pip install langchain_openai

# 預處理資料

In [ ]:
import pandas as pd
import os

def read_taifex(path:str, product: str, product_m_or_w: str):
    '''
        讀取台灣期貨交易所的歷史資料，並選取特定商品代號及月份(週別)的資料

        年月日: str 
        商品代號: str
        商品月份: str

        回傳: DataFrame
    '''
    
    # 讀取 CSV 檔案，並將所有元素轉換成字串
    if os.path.exists(path):
        df = pd.read_csv(path, encoding="BIG5", dtype=str)
    else:
        return pd.DataFrame()
     
    # 移除所有元素的空白
    df = df.map(lambda x: x.strip())

    # 當 '到期月份(週別)' 包含 '/' 時，將 '成交價格' 的值改為 '近月價格' 的值
    df.loc[df['到期月份(週別)'].str.contains('/'), '成交價格'] = df['近月價格']
    
    # 到期月份(週別)中有些資料是202404/202405，代表轉倉，因此分割字串並選取第一個元素並存到新欄位 "商品月份"
    df['商品月份'] = df['到期月份(週別)'].str.split('/').str[0]

    df = df[df["商品代號"] == product]
    df = df[df["商品月份"] == product_m_or_w]

    # 將"交易日期" 和 "交易時間" 合併成一個新的欄位 "交易時間"
    df['Date'] = df['成交日期'] + df['成交時間']

    # 然後，將 "交易時間" 轉換成 datetime 對象
    # 假設 "交易日期" 是 "YYYYMMDD" 的格式，"交易時間" 是 "HHMMSS" 的格式
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d%H%M%S')

    df.set_index('Date', inplace=True)
    df = df.drop(columns=['成交日期'])
    df = df.drop(columns=['成交時間'])

    return df

In [ ]:
import pandas as pd
import os
from datetime import datetime, time
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import mplfinance as mpf

# 假設你的 DataFrame 是 df
def to_kline(interval:str, df:pd.DataFrame):
    '''
    將原始交易資料轉換為 K 線資料

    參數:
        interval: K 線的時間間隔，例如 '5min' 代表 5 分鐘
        df: 原始交易資料，必須包含 '成交數量(B+S)' 和 '成交價格' 兩個欄位

    回傳值
        df_kline: K 線資料，包含 'Open', 'High', 'Low', 'Close', 'Volume' 五個欄位
    '''

    df.index = pd.to_datetime(df.index)  # 確保索引是 datetime 對象
    df['成交數量(B+S)'] = df['成交數量(B+S)'].astype(int)
    df['成交價格'] = df['成交價格'].astype(float)

    # 計算 OHLC
    df_ohlc = df['成交價格'].resample(interval).agg({'Open':'first', 'High':'max', 'Low':'min', 'Close':'last'})

    # 計算 Volume
    df_volume = df['成交數量(B+S)'].resample(interval).sum()

    # 將兩個 DataFrame 合併
    df_kline = pd.concat([df_ohlc, df_volume], axis=1)
    df_kline = df_kline.rename(columns={'成交數量(B+S)': 'Volume'})  # 將 '成交數量(B+S)' 重新命名為 'Volume'

    return df_kline

def fetch_time(dt:datetime, time_start:time=time(8,45), time_end:time=time(13,45), interval:str="5min", symbol:str="MTX"):
    year = dt.strftime('%Y')
    month = dt.strftime('%m')
    day = dt.strftime('%d')

    path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'taifex', f'Daily_{year}_{month}_{day}.csv')
    # print(path)
    
    # 讀取檔案，如果當天的資料不存在，則讀取下一個月的資料(轉倉後的資料)
    df = read_taifex(path, symbol, year + month)
    if df.empty:
        # print("test", (datetime + relativedelta(months=1)).strftime('%Y%m'))
        df = read_taifex(path, symbol, (dt + relativedelta(months=1)).strftime('%Y%m'))
    if df.empty:
        return None
    df_kline = to_kline(interval, df)

    # 轉成固定時間區間
    # df_kline.index = pd.to_datetime(df_kline.index)
    iday = df_kline[df_kline.index.date == dt]
    iday = iday[iday.index.time >= time_start]
    iday = iday[iday.index.time < time_end]

    return iday

def draw_kline(iday:pd.DataFrame):
    '''
        繪製 K 線圖
        參數:
            iday : pd.DataFrame : 股票資料
        回傳:
            kline_path : str : 圖片檔案路徑
    '''
    iday.index = pd.to_datetime(iday.index)
    mc = mpf.make_marketcolors(up='r',down='g',inherit=True)
    s  = mpf.make_mpf_style(base_mpf_style='tradingview',marketcolors=mc)
    if int(iday.shape[0]) == 0:
        return None, None
    
    # 繪製 K 線圖
    fig, axes = mpf.plot(iday.dropna(), type='candle', style=s,
            ylabel='price',
            volume=True,
            returnfig=True)
    plt.close()

    os.makedirs("img", exist_ok=True)
    idx = 0
    while True:
        kline_path = os.path.join("img", "fig" + str(idx) + ".png")
        if not os.path.exists(kline_path):
            fig.savefig(kline_path)
            return kline_path
        idx += 1


    

In [ ]:
from datetime import datetime, time, timedelta
import pandas as pd
from itertools import product

# 創建日期範圍
start_date = "2023-10-27"
end_date = "2024-06-30"
date_range = pd.date_range(start=start_date, end=end_date)

# 對於每一天的每一小時調用 fetch_time
# folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
batch_size = 15

# 設定讀取日期範圍
date_range = pd.date_range(start_date, end_date, freq='D')
time_range = [time(9, 0), time(10, 0), time(11, 0), time(12, 0), time(13, 0)]
datetimes = [datetime.combine(date, time) for date, time in product(date_range, time_range)]

# 讀取後退提示 k 
with open('SBP_candidate/kline3.txt', 'r', encoding='utf-8') as f:
    content = f.read()


datetimes_idx = 0
dts = []
for idx, dt in enumerate(datetimes):
    date = dt.date()
    start_time = dt.time()
    end_time = (dt + timedelta(hours=1)).time()
    df = fetch_time(date, start_time, end_time, "5min", "MTX")

    if df is None or len(df)<1:
        continue
    # change = df['Close'].iloc[-1]/df['Open'].iloc[0]

    # path = draw_kline(df)
    # if path is None:
    #     continue
    
    date_str = date.strftime("%Y%m%d")
    start_time_str = start_time.strftime("%H")
    end_time_str = end_time.strftime("%H")
    formatted_str = f"{date_str}_{start_time_str}_{end_time_str}"
        
    df.to_csv("temp_data/" + formatted_str + ".csv")


# 主程式

In [ ]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain.schema.messages import HumanMessage
from langchain.schema.messages import SystemMessage
from langchain.schema.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_openai import OpenAI

from IPython.display import display
from IPython.display import Markdown



def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

# model = "gpt-3.5-turbo-instruct"
# model = "gpt-4o"
# temperature = 1
# top_p = 0.9
# seed = 0 

# llm = OpenAI(
#                   openai_api_key=os.getenv('OPENAI_API_KEY'),
#                   model_name=model
#                   )


model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

In [ ]:
from datetime import datetime, time
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import timedelta
import json


def step1():
    path_price = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'mtx_1h')
    files = os.listdir(path_price)
    files.sort()
    files = files[:50]

    # 讀取 k 
    with open('SBP_candidate/kline3.txt', 'r', encoding='utf-8') as f:
        sbp_response = f.read()

    batch_size = 10
    file_id = 0
    file_pred = []
    while file_id < len(files):
        batch = []

        for i in range(batch_size):
            file = files[file_id+i]
            if file.split('_')[2].find("14") == 0: # 當天最後一筆不預測
                continue
            df = pd.read_csv(path_price + "/" + file)

            # print(df)
            path_klinefig = draw_kline(df)
            if path_klinefig is None:
                continue

            file_pred.append(file)

            # 設定對話內容
            messages = [
                SystemMessage(content="你是一個厲害的股市操盤手，專長是當日沖消"),
                HumanMessage(content="k線對於市場情緒的概念是什麼?")
            ]
            res1 = AIMessage(content=sbp_response  , response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-3bdfe58e-7707-4a1b-8f35-b8953592f5ff-0')
            messages.append(res1)
            messages.append(
                HumanMessage(
                    content=[
                        {
                            "type": "text",
                            "text": "這是一張小台指的5分k線圖，請用上面的概念，並透過這張圖的具體形態，關鍵價格，成交量的變化趨勢，預測接下來一小時的市場情緒狀態，如果市場情緒好則只能回答 #long、如果不好則只能回答 #short，如果情緒中立則只能回答 #hold",
                        },
                        {"type": "image_url", "image_url": path_klinefig}  # 替換為實際圖像路徑
                    ]
                )
            )
            batch.append(messages)  

        responses = llm.batch(batch)
        print(responses)

        out_path = os.path.abspath(os.getcwd()) + "/out_kline_SBP/3/"
        os.makedirs(out_path, exist_ok=True)
        for i in range(len(responses)):
            output_data = {
                "model": model,
                "temperature": temperature,
                "top_p": top_p,
                "seed": seed,
                # "token": responses[i].usage_metadata,
                # "raw_data": responses[i].content,
                "raw_data": responses[i],
            }

            json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
            if not os.path.exists(out_path):
                os.makedirs(out_path)
            with open(out_path + file_pred[i].split('.')[0] + ".json" , "w", encoding="UTF-8") as f: # 寫入預測結果
                f.write(json_content)

        break
        # time.sleep(60)
        datetimes_idx += batch_size


In [ ]:
step1()

In [ ]:
import pandas as pd
import json
import os

# 計算準確率和平均報酬率
gain = []
loss = []

tp = 0
fp = 0
tn = 0
fn = 0

max_gain = 0
max_loss = 0

path = "out_kline_SBP/3/"
files = os.listdir(path)

token_input = 0
token_output = 0

# print(len(files))
files = sorted(files)

for idx, file in enumerate(files):

    # print(file)
    with open(path + file, "r", encoding='utf-8') as f:
        data = json.load(f)
        # token_input += data["token"]["input_tokens"]
        # token_output += data["token"]["output_tokens"]
        
        sig_str = data["raw_data"]

        sig = 0
        if sig_str.find("hold") > 0:
            sig = 0
        elif sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0

    if idx+1 == len(files):
        break
    # print(files[idx+1])
    path_price = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'mtx_1h')
    # 找到該檔案名稱的索引
    index = files.index(file)
    df = pd.read_csv(path_price+ "/" + files[index+1].split('.')[0] + ".csv")
    rtn = 1- (df['Close'].iloc[-1]/df['Open'].iloc[0])

    if pd.isna(rtn):
        rtn = 0

    if sig > 0:
        if rtn > 0:
            tp += 1
            gain.append(rtn)
            if rtn > max_gain:
                max_gain = rtn

        else:
            fp += 1
            loss.append(rtn)
            if rtn < max_loss:
                max_loss = rtn


    elif sig == -1: 
        if rtn < 0:
            tn += 1
            gain.append(-rtn)
            if -rtn > max_gain:
                max_gain = -rtn

        else:
            fn += 1
            loss.append(-rtn)
            if -rtn < max_loss:
                max_loss = -rtn

print(f'tp {tp}, fp {fp}, tn {tn}, fn {fn}')
accuracy = (tp + tn) / (tp + fp + tn + fn)
print(f'Accuracy:{accuracy}, Expected value:{(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')
print(f'Average gain: {sum(gain) / len(gain)}, Average loss: {sum(loss) / len(loss)}')
print(f'Max gain: {max_gain}, Max loss: {max_loss}')
print(f'詢問次數 {len(files)}')
print()

# print("--------------------")
# print(f'token: input {token_input}, output {token_output}')
# print(f'累計預估費用 {(token_input/1000000)*0.35 + (token_output/1000000)*1.05} USD, {((token_input/1000000)*0.35 + (token_output/1000000)*1.05)*32.6} TWD')